# TP - Variational Autoencoder (VAE) Convolucional
### Redes Neuronales / Deep Learning

**Trabajo en grupos de 2 personas.** Pueden consultarse entre grupos.

**Defensa oral: 40 minutos por grupo.**

Esta es la **Notebook 2 de 3**. Asumimos que ya hicieron la Notebook 1 (Autoencoder), porque acá reusamos varias piezas (encoder/decoder convolucional, perceptual loss, TensorBoard, MLflow, Optuna) y nos enfocamos en lo nuevo: el **espacio latente probabilístico** y la posibilidad de **generar** imágenes nuevas.

## Objetivos
- Entender la diferencia entre un Autoencoder y un VAE.
- Implementar el *reparametrization trick*.
- Combinar loss de reconstrucción (MSE + perceptual) con la divergencia KL.
- Generar imágenes nuevas e interpolar en el espacio latente.


## 0. Setup

In [ ]:
!pip install -q mlflow optuna


In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import make_grid
from torchvision.models import vgg16, VGG16_Weights
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import mlflow
import mlflow.pytorch
import optuna

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)


## 1. Datos: CIFAR-10 (igual que en la Notebook 1)

In [ ]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## 2. ¿Qué diferencia a un VAE de un Autoencoder?

En el Autoencoder, el encoder produce un único vector `z` determinístico. En el VAE, el encoder produce los parámetros de una **distribución normal** en el espacio latente: una media `mu` y una log-varianza `logvar`. Para generar `z` muestreamos de esa distribución, pero de una forma que siga siendo diferenciable (**reparametrization trick**):

```
z = mu + sigma * epsilon,   con epsilon ~ N(0, 1)
```

Esto permite que el gradiente fluya a través de `mu` y `sigma`, y que el espacio latente quede "organizado" (continuo), lo que nos permite generar muestras nuevas simplemente sampleando de una `N(0,1)` y decodificando.

### Encoder y Decoder (ya resueltos, igual que en la Notebook 1 salvo por las dos cabezas mu/logvar)

In [ ]:
class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(128 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(128 * 4 * 4, latent_dim)

    def forward(self, x):
        h = self.conv(x)
        h = h.view(h.size(0), -1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128 * 4 * 4)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1), nn.Sigmoid(),
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(x.size(0), 128, 4, 4)
        x = self.deconv(x)
        return x


### TODO 1: completar el reparametrization trick

Dado `mu` y `logvar` (ojo: es log-varianza, no desviación estándar directamente), completar `reparameterize`.

Pista: `sigma = exp(0.5 * logvar)`, y `epsilon` se samplea con `torch.randn_like(sigma)`.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.encoder = VAEEncoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def reparameterize(self, mu, logvar):
        # TODO: implementar z = mu + sigma * epsilon
        sigma = None       # calcular a partir de logvar
        epsilon = None     # samplear ruido con la misma forma que sigma
        z = None           # combinar mu, sigma y epsilon
        return z

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar


In [ ]:
# Sanity check
_dummy = torch.randn(4, 3, 32, 32)
_vae_check = VAE(latent_dim=128)
_x_hat, _mu, _logvar = _vae_check(_dummy)
assert _x_hat is not None and _x_hat.shape == _dummy.shape, "Revisá reparameterize/forward"
print("Shapes OK:", _x_hat.shape, _mu.shape, _logvar.shape)


## 3. Loss del VAE: reconstrucción + KL divergence

La loss total es:

```
loss = reconstruction_loss(x, x_hat)  +  beta * KL(q(z|x) || N(0,1))
```

Donde la divergencia KL entre la distribución aproximada `N(mu, sigma^2)` y una normal estándar `N(0,1)` tiene forma cerrada:

```
KL = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
```

`beta` es un hiperparámetro que pesa cuánto "empujamos" el espacio latente a parecerse a una normal estándar (a mayor beta, espacio más organizado pero reconstrucciones más borrosas — este trade-off se llama **beta-VAE**).

Para la reconstrucción reusamos la perceptual loss + MSE de la Notebook 1.

### TODO 2: completar la loss del VAE

In [ ]:
class PerceptualLoss(nn.Module):
    def __init__(self, layers=8):
        super().__init__()
        vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features[:layers]
        for p in vgg.parameters():
            p.requires_grad = False
        self.vgg = vgg.eval().to(device)

    def forward(self, x_hat, x):
        return nn.functional.mse_loss(self.vgg(x_hat), self.vgg(x))


def vae_loss(x_hat, x, mu, logvar, perceptual_loss_fn, lam=0.1, beta=1.0):
    pixel_loss = nn.functional.mse_loss(x_hat, x, reduction="mean")
    perc_loss = perceptual_loss_fn(x_hat, x)
    recon_loss = pixel_loss + lam * perc_loss

    # TODO: completar la fórmula de KL divergence de la celda de arriba.
    # Ojo: la fórmula clásica sale sumada sobre las dimensiones latentes y el batch,
    # y después conviene normalizar dividiendo por x.size(0) (tamaño de batch)
    # para que quede en una escala comparable a recon_loss.
    kl = None

    total = recon_loss + beta * kl
    return total, recon_loss, kl


In [ ]:
# Sanity check de la loss completa
perceptual_loss_fn = PerceptualLoss()
_a = torch.rand(2, 3, 32, 32).to(device)
_b = torch.rand(2, 3, 32, 32).to(device)
_mu = torch.randn(2, 128).to(device)
_logvar = torch.randn(2, 128).to(device)
_total, _recon, _kl = vae_loss(_a, _b, _mu, _logvar, perceptual_loss_fn, lam=0.1, beta=1.0)
print("total:", _total, "| recon:", _recon, "| kl:", _kl)
assert _total is not None, "Completá el cálculo de KL en vae_loss"


## 4. Entrenamiento (TensorBoard + MLflow, ya resuelto)

In [ ]:
%load_ext tensorboard


In [ ]:
def train_vae(model, loader, optimizer, perceptual_loss_fn, epochs, lam=0.1, beta=1.0,
              writer=None, run_name="vae", log_every=1):
    model.to(device)
    history = []
    for epoch in range(epochs):
        model.train()
        running_total, running_recon, running_kl = 0.0, 0.0, 0.0
        for imgs, _ in loader:
            imgs = imgs.to(device)
            optimizer.zero_grad()
            x_hat, mu, logvar = model(imgs)
            total, recon, kl = vae_loss(x_hat, imgs, mu, logvar, perceptual_loss_fn, lam=lam, beta=beta)
            total.backward()
            optimizer.step()
            running_total += total.item() * imgs.size(0)
            running_recon += recon.item() * imgs.size(0)
            running_kl += kl.item() * imgs.size(0)

        n = len(loader.dataset)
        avg_total, avg_recon, avg_kl = running_total / n, running_recon / n, running_kl / n
        history.append(avg_total)
        print(f"[{run_name}] Epoch {epoch+1}/{epochs} - loss_total: {avg_total:.4f} "
              f"(recon: {avg_recon:.4f}, kl: {avg_kl:.4f})")

        if writer is not None:
            writer.add_scalar("loss/total", avg_total, epoch)
            writer.add_scalar("loss/recon", avg_recon, epoch)
            writer.add_scalar("loss/kl", avg_kl, epoch)
        mlflow.log_metric("loss_total", avg_total, step=epoch)
        mlflow.log_metric("loss_recon", avg_recon, step=epoch)
        mlflow.log_metric("loss_kl", avg_kl, step=epoch)

        if writer is not None and epoch % log_every == 0:
            model.eval()
            with torch.no_grad():
                sample, _ = next(iter(loader))
                sample = sample[:8].to(device)
                recon_imgs, _, _ = model(sample)
                grid = make_grid(torch.cat([sample, recon_imgs]), nrow=8)
                writer.add_image("originales_vs_reconstruidas", grid, epoch)

                z = torch.randn(8, model.encoder.fc_mu.out_features).to(device)
                generated = model.decoder(z)
                writer.add_image("generadas_desde_ruido", make_grid(generated, nrow=8), epoch)
    return history


def evaluate_vae(model, loader, perceptual_loss_fn, lam=0.1, beta=1.0):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device)
            x_hat, mu, logvar = model(imgs)
            total, _, _ = vae_loss(x_hat, imgs, mu, logvar, perceptual_loss_fn, lam=lam, beta=beta)
            total_loss += total.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


## 5. Búsqueda de hiperparámetros con Optuna

Ahora buscamos también `beta` (el peso de la KL), además de `lr` y `latent_dim`.

In [ ]:
def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    latent_dim = trial.suggest_categorical("latent_dim", [32, 64, 128])
    beta = trial.suggest_float("beta", 0.1, 2.0)

    model = VAE(latent_dim=latent_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    train_vae(model, train_loader, optimizer, perceptual_loss_fn,
              epochs=3, beta=beta, writer=None, run_name=f"trial_{trial.number}")
    return evaluate_vae(model, test_loader, perceptual_loss_fn, beta=beta)


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5)

print("Mejores hiperparámetros:", study.best_params)
print("Mejor loss de validación:", study.best_value)


## 6. Entrenamiento final

In [ ]:
best_params = study.best_params
LATENT_DIM = best_params["latent_dim"]
LR = best_params["lr"]
BETA = best_params["beta"]
EPOCHS = 20
LAMBDA_PERCEPTUAL = 0.1

model = VAE(latent_dim=LATENT_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
writer = SummaryWriter(log_dir="runs/vae_final")

mlflow.set_experiment("vae_cifar10")
with mlflow.start_run(run_name="vae_final"):
    mlflow.log_params({
        "latent_dim": LATENT_DIM, "lr": LR, "beta": BETA, "epochs": EPOCHS,
        "batch_size": BATCH_SIZE, "lambda_perceptual": LAMBDA_PERCEPTUAL,
    })
    history = train_vae(model, train_loader, optimizer, perceptual_loss_fn,
                         epochs=EPOCHS, lam=LAMBDA_PERCEPTUAL, beta=BETA, writer=writer)
    mlflow.pytorch.log_model(model, "model")

writer.close()


In [ ]:
%tensorboard --logdir runs


## 7. Generación de imágenes nuevas

In [ ]:
model.eval()
with torch.no_grad():
    z = torch.randn(16, LATENT_DIM).to(device)
    generated = model.decoder(z)

grid = make_grid(generated, nrow=8)
plt.figure(figsize=(10, 3))
plt.imshow(grid.cpu().permute(1, 2, 0))
plt.axis("off")
plt.title("Imágenes generadas desde ruido")
plt.show()


## 8. Interpolación en el espacio latente

Codificamos dos imágenes reales, y generamos una secuencia interpolando linealmente entre sus vectores latentes.

In [ ]:
model.eval()
with torch.no_grad():
    imgs, _ = next(iter(test_loader))
    img_a, img_b = imgs[0:1].to(device), imgs[1:2].to(device)
    mu_a, _ = model.encoder(img_a)
    mu_b, _ = model.encoder(img_b)

    steps = 8
    alphas = torch.linspace(0, 1, steps).to(device)
    interpolations = torch.cat([model.decoder(mu_a * (1 - a) + mu_b * a) for a in alphas])

grid = make_grid(interpolations, nrow=steps)
plt.figure(figsize=(12, 2))
plt.imshow(grid.cpu().permute(1, 2, 0))
plt.axis("off")
plt.title("Interpolación en el espacio latente")
plt.show()


## 9. Preguntas de reflexión (para la defensa)

1. ¿Qué diferencia observaron entre las reconstrucciones del Autoencoder (Notebook 1) y las del VAE? ¿Cuál es más borrosa y por qué?
2. ¿Qué pasó con las imágenes generadas al aumentar mucho `beta`? ¿Y al bajarlo cerca de 0?
3. ¿Qué observan en la interpolación? ¿Las transiciones son suaves o abruptas? ¿Qué dice esto sobre la organización del espacio latente?
4. ¿Por qué el Autoencoder de la Notebook 1 no sirve para generar imágenes nuevas de forma confiable, y el VAE sí?
5. ¿Qué mejoraron/empeoraron los hiperparámetros que encontró Optuna respecto a valores "a mano" que se les ocurra probar?
